# Lilly v2 — listener pass-2 (whisper-large-v3, wider mix)

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian speech only; train on Kaggle,
measure before/after inside this notebook only.

Pass-1 (Kaggle v12) was whisper-small on FLEURS hr only: 6,521 rows, 47% Bosnian,
38.4% → 33.9% WER. There is no larger official Whisper than large-v3. This pass
changes the base **and** widens the Croatian side: FLEURS hr (read speech) plus
voxpopuli_hr (EP floor, CC0 + attribution). ParlaSpeech-HR is CC BY-SA and is
not pulled. Bosnian share stays **0.47** — the mixer repeats Bosnian so the extra
Croatian cannot drown it. Two epochs, not three: v14 measured 6,521×3 ≈ 7h22m
train on large-v3; a wider mix at 3 epochs misses the 12h session wall.

BEFORE and AFTER both convert the **same** large-v3 checkpoint. A comparison
against whisper-small would not be this run.

Set these in the panel on the right:

- **Session options → Accelerator → GPU T4**
- **Session options → Internet → On**

Then **Save Version → Save & Run All (Commit)** and close the tab.


In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co", "https://datasets-server.huggingface.co"):
    reachable(host)
print("network ok")

def run(*cmd):
    """Run a step and let a failure actually stop the notebook."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)


In [ ]:
# 2. Get the Lilly code
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Lilly"], check=True)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert Path("/kaggle/working/Lilly/training").is_dir(), "clone produced nothing"
os.chdir("/kaggle/working/Lilly")
print("working in", os.getcwd())


In [ ]:
# 3. Install what we need (~3 min)
# The versions are read out of the repo's own requirements.txt rather than
# copied into this cell. A second, hand-kept list is exactly how the last run
# died: peft is pinned in requirements.txt and was simply absent from here, so
# Kaggle's own much newer peft got used instead — and that one's torchao
# dispatcher raises against the torchao Kaggle also ships. The first
# get_peft_model() call blew up, after a 3 GB download and forty minutes.
NEEDED = ["transformers", "accelerate", "peft", "faster-whisper", "ctranslate2",
          "soundfile", "scipy", "pyarrow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
unpinned = [n for n in NEEDED if n not in pins]
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin in requirements.txt, taking latest:", unpinned or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])


In [ ]:
# 3b. Prove this machine can actually train, before an hour is spent finding out
# Two things have to hold and neither shows up in a version number: peft must be
# able to build a LoRA layer on this image, and the GPU Kaggle handed us must
# actually run kernels. The last run satisfied "GPU is available" and still could
# not compute — Kaggle gave a P100 (sm_60) that the installed PyTorch does not
# support, and separately peft could not build a layer at all.
#
# So: build a real LoRA layer, put it on the GPU, push a gradient through it. It
# takes about twenty seconds and it fails here, loudly, instead of after the
# download.
import torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__)

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(32, 32)
    def forward(self, x):
        return self.q_proj(x)

tiny = get_peft_model(Tiny(), LoraConfig(r=4, target_modules=["q_proj"]))
try:
    tiny = tiny.cuda()
    out = tiny(torch.randn(4, 32, device="cuda")).sum()
    out.backward()
except RuntimeError as exc:
    raise SystemExit(
        f"The GPU cannot run this build of PyTorch ({exc}).\n"
        f"Card: {torch.cuda.get_device_name(0)}. Right panel -> Session options ->"
        f" Accelerator -> GPU T4 x2, then Save & Run All again.") from exc

lora = [n for n, p in tiny.named_parameters() if "lora_" in n and p.grad is not None]
assert lora, "peft built a LoRA layer but no gradient reached it"
print(f"LoRA trains on {torch.cuda.get_device_name(0)}: "
      f"{len(lora)} adapter tensors took a gradient")
del tiny
torch.cuda.empty_cache()


In [ ]:
# 4. Download the speech: clips to train on, and clips held back to judge with (~3 GB)
# The audio goes to scratch space, not into /kaggle/working — everything in the working
# directory is copied into the version Output, and 3 GB of wav files there is waste.
scratch = Path("/kaggle/temp/speech" if Path("/kaggle/temp").is_dir() else "/tmp/lilly-speech")
scratch.mkdir(parents=True, exist_ok=True)
if Path("data/speech").is_symlink():
    Path("data/speech").unlink()          # re-running the cell should not fail
if not Path("data/speech").exists():
    Path("data/speech").symlink_to(scratch)

run("python3", "data/scripts/download_speech_data.py")

for split in ("train", "valid", "test"):
    n = sum(1 for _ in open(f"data/speech/{split}.tsv", encoding="utf-8"))
    assert n > 100, f"{split}.tsv has only {n} clips — a download failed"
    print(f"{split}: {n:,} clips")


In [ ]:
# 4b. Croatian speech — downloaded here rather than uploaded from home.
# Kaggle's connection runs at roughly twenty times the one this project is
# developed on, and the audio is several gigabytes: pulling it here costs
# minutes where uploading it as a dataset would cost most of an evening.
# Scratch space for the same reason the Bosnian clips use it — anything left in
# /kaggle/working is copied into the Output, and gigabytes of wav there is waste.
extra = Path("/kaggle/temp/speech-extra" if Path("/kaggle/temp").is_dir()
             else "/tmp/lilly-speech-extra")
extra.mkdir(parents=True, exist_ok=True)
if Path("data/speech-extra").is_symlink():
    Path("data/speech-extra").unlink()
if not Path("data/speech-extra").exists():
    Path("data/speech-extra").symlink_to(extra)

# Two sources, two calls: a single --hours would override both defaults.
# fleurs_hr default is already 12 h (the whole train split). voxpopuli_hr
# default is 8 h of spontaneous EP speech FLEURS does not have.
run("python3", "data/scripts/download_extra_speech.py",
    "--source", "fleurs_hr", "--hours", "12")
run("python3", "data/scripts/download_extra_speech.py",
    "--source", "voxpopuli_hr")

fleurs_n = sum(1 for _ in open("data/speech-extra/fleurs_hr/train.tsv", encoding="utf-8"))
vox_path = Path("data/speech-extra/voxpopuli_hr/train.tsv")
assert fleurs_n > 500, f"only {fleurs_n} FLEURS hr clips — the download did not work"
assert vox_path.is_file(), (
    "voxpopuli_hr missing — this pass is the wider mix, not another FLEURS-only run")
vox_n = sum(1 for _ in open(vox_path, encoding="utf-8"))
assert vox_n > 200, f"only {vox_n} voxpopuli clips — the second source did not land"
print(f"Croatian FLEURS hr: {fleurs_n:,}  voxpopuli_hr: {vox_n:,}")


In [ ]:
# 4c. Mix them, keeping Bosnian at a stated share of the examples.
# Concatenating is the obvious move and the wrong one: with far more Croatian
# than Bosnian the model hears mostly Croatian and drifts towards it, and the
# overall error rate can fall while Bosnian gets worse.
#
# Pass-1 asked for 0.35 and landed at 47% because FLEURS hr was already that
# large a slice. This pass has more Croatian, so 0.35 would actually drown
# Bosnian. 0.47 is pass-1's measured mix and the floor the mixer holds by
# repeating Bosnian clips.
BOSNIAN_SHARE = 0.47
# v14: 6,521 rows × 3 epochs = 7h 22m train on large-v3 (~2,650 row-epochs/h).
# A 12h session minus download + BEFORE/AFTER WER leaves ~9.5h of train
# (~25,000 row-epochs). Two epochs on the wider mix; three would miss the wall.
SPEECH_EPOCHS = 2
SPEECH_BASE = "openai/whisper-large-v3"

run("python3", "data/scripts/build_speech_mix.py", "--share", str(BOSNIAN_SHARE))

MIX = "data/speech-extra/train-mix.tsv"
bosnian_only = sum(1 for _ in open("data/speech/train.tsv", encoding="utf-8"))
mixed = sum(1 for _ in open(MIX, encoding="utf-8"))
assert mixed > bosnian_only, (
    f"the mix has {mixed:,} rows against {bosnian_only:,} Bosnian — no Croatian "
    f"got in, so this run would not test what it is here to test")
budget = mixed * SPEECH_EPOCHS
assert budget <= 26_000, (
    f"{mixed:,} rows × {SPEECH_EPOCHS} epochs = {budget:,} row-epochs; "
    f"v14's clock says that misses the 12h wall. Cut voxpopuli hours, "
    f"do not silently drop epochs.")
print(f"mixed: {mixed:,} rows, from {bosnian_only:,} Bosnian, "
      f"{SPEECH_EPOCHS} epochs, {budget:,} row-epochs (budget 26,000)")


In [ ]:
# 5. BEFORE: how bad is the untrained listener on Bosnian it has never heard?
# Same format, same measure as the after-run, so the two numbers are comparable.
run("python3", "training/train_speech.py", "--base", SPEECH_BASE,
    "--convert-only", "/kaggle/working/listen-before")
run("python3", "training/evaluate_speech.py", "--data", "data/speech/test.tsv",
    "--model", "/kaggle/working/listen-before", "--limit", "200", "--show", "3")


In [ ]:
# 6. THE REAL TRAINING (~8-10 hours on T4 for large-v3 × wider mix × 2 epochs)
# --no-convert on purpose: the trained checkpoint is saved and packaged in the next
# cell before anything else can go wrong with it.
#
# large-v3 writes sharded weights (model-00001-of-0000N.safetensors), NOT a single
# model.safetensors — asserting that filename after a successful 7h train killed v14.
run("python3", "training/train_speech.py", "--data", MIX, "--base", SPEECH_BASE,
    "--epochs", str(SPEECH_EPOCHS), "--batch-size", "1", "--grad-accum", "16",
    "--no-convert")
trained = Path("models/lilly/listen-trained")
weight_files = (
    list(trained.glob("model*.safetensors"))
    + list(trained.glob("pytorch_model*.bin"))
    + list(trained.glob("*.safetensors"))
)
assert weight_files, (
    f"nothing was trained under {trained}: "
    f"{sorted(p.name for p in trained.iterdir()) if trained.is_dir() else 'missing'}"
)
print("trained weights:", ", ".join(p.name for p in weight_files))
# Zip immediately so convert/eval failures cannot erase a finished train.
run("zip", "-qr", "/kaggle/working/lilly-listen-trained.zip",
    "models/lilly/listen-trained")
print("saved /kaggle/working/lilly-listen-trained.zip before convert")


In [ ]:
# 7. Convert the trained checkpoint into the format the app loads
run("python3", "training/train_speech.py",
    "--base", "models/lilly/listen-trained", "--convert-only", "models/lilly/listen")
assert Path("models/lilly/listen/model.bin").is_file(), "conversion produced nothing"


In [ ]:
# 8. AFTER: the same clips, the same measure. Lower is better.
run("python3", "training/evaluate_speech.py", "--data", "data/speech/test.tsv",
    "--limit", "200", "--show", "3")


In [ ]:
# 9. Package both listeners — the trained one, and the untrained one to fall back to
run("zip", "-qr", "/kaggle/working/lilly-listen.zip", "models/lilly/listen")
size = Path("/kaggle/working/lilly-listen.zip").stat().st_size
assert size > 1_000_000, f"the zip is only {size} bytes"
print(f"lilly-listen.zip — {size / 1048576:.0f} MB")
print("listen-before/ is in the Output too, if you need to go back")


**Done.** Compare the error rates printed by cell 5 and cell 8 — same clips, same
measure, so the difference is real.

If it dropped, download `lilly-listen.zip` from the **Output** tab and unzip it over
`models/lilly/listen/`. Keep a copy of the listener you are replacing first; the app has
no other way back.

If it did not drop, keep what you have. More clips, or more epochs, before another run.
